### Сравнение моделей (Model Comparison), несколько resist_id — **calc_* lag1–lag6 и lag12**

Вариант [`7_model_comparison_multi.ipynb`](7_model_comparison_multi.ipynb): вместо полной календарной развёртки `calc_k__YYYY_MM` в наборах **B/C** используются признаки **`calc_k_lag1` … `calc_k_lag6`** и **`calc_k_lag12`** (сдвиги `calculate_parameter` на 1–6 и 12 месяцев по `operation_id`; см. `DEFAULT_CALC_SHIFTS` в `scripts/calc_lag_features.py`).

Эксперимент: обучить **табличные и per-series модели** на **6 наборах признаков** (только табличные) и сравнить метрики для **списка** `RESIST_IDS` за один прогон.

В **сводной таблице** в ноутбуке показывается по каждому `resist_id` только **топ-5** по метрике «Качество»; полные результаты — в `df_results_full`.

В **Excel** лист `{id} **Итог**` — **топ-20** строк на id (как ожидает `model_comparison_analytics.py` для блоков «топ-20»); остальные два листа без изменений. Файл: `model_comparison_multi_calc_lag12.xlsx`.

Остальное — как в `7_model_comparison_multi.ipynb` (табличные модели, per-series, тест июль–декабрь 2025).


In [ ]:
import json
import sys
import time
import logging
from pathlib import Path

_ROOT = Path("..").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
from scripts.calc_lag_features import (
    add_calc_shift_lags,
    calc_lag_columns_for_sets,
    DEFAULT_CALC_SHIFTS,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor,
)

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.forecasting.theta import ThetaModel
from statsmodels.tsa.statespace.sarimax import SARIMAX

HAS_PROPHET = True
try:
    from prophet import Prophet
    logging.getLogger("prophet").setLevel(logging.WARNING)
    logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
except ImportError:
    HAS_PROPHET = False
    print("prophet не установлен — конфигурации Prophet будут пропущены")

HAS_PMDARIMA = True
try:
    import pmdarima  # noqa: F401
except ImportError:
    HAS_PMDARIMA = False
    print("pmdarima не установлен — конфигурация AutoARIMA будет пропущена")

%matplotlib inline
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning)

#### Конфигурация

In [ ]:
RESIST_ID = 124  # не используется, если задан список RESIST_IDS
RESIST_IDS = [124]  # список resist_id; дубликаты убираются автоматически
SUMMARY_TOP_PER_RID = 5  # строк в сводке в ноутбуке (на каждый resist_id)
EXCEL_ITOG_TOP = 20  # строк на лист «Итог» в Excel (совместимость с аналитикой топ-20)

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]

# Набор C: только ключи из calculate_parameter, совпадающие с flatten в CSV; входы модели нагрузки — excel_workload_model.md (C4,C14,C15,C16,C6,C7,C9).
CALC_WHITELIST = [
    "calc_workDayMonth_2",
    "calc_hourInWeek_12",
    "calc_numberWorkHour_1",
    "calc_effective_13",
    "calc_absenteeism_14",
    "calc_overtime_4",
    "calc_cross_5",
    "calc_employFact_7",
]

LAG_FEATURES = ["lag_1", "lag_12", "rolling_mean_3"]
OUT_XLSX = Path("../data/processed/model_comparison_multi_calc_lag12.xlsx")


#### Загрузка данных и подготовка (с calc_flat)

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(f"Файлы данных не найдены в {RAW_DIR}")

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record, **calc_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = pd.to_numeric(df_model39[col], errors="coerce")
print(f"calc_* колонки ({len(calc_cols)}):", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

RESIST_IDS = list(dict.fromkeys(RESIST_IDS))
avail = set(int(x) for x in df_model39["resist_id"].dropna().unique())
missing = [r for r in RESIST_IDS if r not in avail]
if missing:
    sample = sorted(avail)[:15]
    raise ValueError(
        f"Нет данных для resist_id={missing}. Примеры доступных id: {sample}..."
    )
print(f"RESIST_IDS ({len(RESIST_IDS)}): {RESIST_IDS}")


#### Feature Engineering: lag_1, lag_12, rolling_mean_3; calc_* — lag1–lag6 и lag12

In [ ]:
# Лаги и наборы признаков строятся в ячейке «Табличные + per-series» для каждого resist_id.


#### Проверка пропусков

In [ ]:
# Проверка пропусков выполняется в цикле обучения для каждого df.


#### Наборы признаков (Feature Sets) и split

In [ ]:
# FEATURE_SETS и train/test split задаются в цикле по RESIST_IDS.


#### Метрики + вспомогательные функции

In [ ]:
def calc_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0, 100,
        )
    tochnost = round(float(np.mean(acc)), 2)
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = round(max(0.0, (1 - cv)) * 100, 2)
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = round(2 * tochnost * stab / (tochnost + stab), 2)
    return tochnost, stab, kachestvo

#### Табличные модели: 17 конфигураций × 6 наборов признаков = 102 комбинации

In [ ]:
def train_ets_per_series(
    df_full,
    mask_train,
    mask_test,
    trend="add",
    damped_trend=False,
    seasonal=None,
    seasonal_periods=None,
):
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        try:
            if seasonal is not None and seasonal_periods is not None:
                min_hist = max(4, 2 * int(seasonal_periods))
                if len(y_hist) < min_hist:
                    raise ValueError("too short for seasonal")
            elif len(y_hist) < 4:
                raise ValueError("too short")
            if trend == "mul" and np.any(y_hist <= 0):
                raise ValueError("mul trend requires positive series")
            if seasonal == "mul" and np.any(y_hist <= 0):
                raise ValueError("mul seasonal requires positive series")
            model = ExponentialSmoothing(
                y_hist,
                trend=trend,
                damped_trend=damped_trend,
                seasonal=seasonal,
                seasonal_periods=seasonal_periods if seasonal is not None else None,
            )
            fit = model.fit(optimized=True, use_brute=False)
            fc = fit.forecast(n_fc)
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_prophet_per_series(df_full, mask_train, mask_test, prophet_params):
    if not HAS_PROPHET:
        return None
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        try:
            if len(y_hist) < 4:
                raise ValueError("too short")
            prophet_df = pd.DataFrame({
                "ds": pd.to_datetime(train_part["date"]),
                "y": y_hist,
            })
            m = Prophet(**prophet_params)
            m.fit(prophet_df)
            future = m.make_future_dataframe(periods=n_fc, freq="MS")
            fc = m.predict(future)["yhat"].iloc[-n_fc:].values
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_theta_per_series(df_full, mask_train, mask_test):
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        try:
            if len(y_hist) < 4:
                raise ValueError("too short")
            tm = ThetaModel(y_hist, period=12)
            res = tm.fit()
            fc = res.forecast(n_fc)
            fc = np.asarray(fc, dtype=float).ravel()
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_sarimax_per_series(df_full, mask_train, mask_test):
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        try:
            if len(y_hist) < 8:
                raise ValueError("too short")
            if len(y_hist) >= 24:
                model = SARIMAX(
                    y_hist,
                    order=(1, 1, 1),
                    seasonal_order=(1, 1, 1, 12),
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                )
                fit = model.fit(disp=False)
                fc = fit.forecast(n_fc)
            else:
                model = SARIMAX(
                    y_hist,
                    order=(1, 1, 1),
                    seasonal_order=(0, 0, 0, 0),
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                )
                fit = model.fit(disp=False)
                fc = fit.forecast(n_fc)
            fc = np.asarray(fc, dtype=float).ravel()
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_auto_arima_per_series(df_full, mask_train, mask_test):
    from pmdarima import auto_arima

    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        try:
            if len(y_hist) < 8:
                raise ValueError("too short")
            aa = auto_arima(
                y_hist,
                seasonal=True,
                m=12,
                suppress_warnings=True,
                error_action="ignore",
                stepwise=True,
                max_p=3,
                max_q=3,
                max_P=2,
                max_Q=2,
            )
            fc = aa.predict(n_periods=n_fc)
            fc = np.asarray(fc, dtype=float).ravel()
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


SERIES_CONFIGS = {
    "ETS-add": {"fn": "ets", "trend": "add", "damped_trend": False, "seasonal": None, "seasonal_periods": None},
    "ETS-damped": {"fn": "ets", "trend": "add", "damped_trend": True, "seasonal": None, "seasonal_periods": None},
    "ETS-seasonal-add": {
        "fn": "ets",
        "trend": "add",
        "damped_trend": False,
        "seasonal": "add",
        "seasonal_periods": 12,
    },
    "ETS-seasonal-damped": {
        "fn": "ets",
        "trend": "add",
        "damped_trend": True,
        "seasonal": "add",
        "seasonal_periods": 12,
    },
    "ETS-mul-trend": {"fn": "ets", "trend": "mul", "damped_trend": False, "seasonal": None, "seasonal_periods": None},
    "Prophet-default": {
        "fn": "prophet",
        "params": {"seasonality_mode": "additive", "n_changepoints": 10,
                   "yearly_seasonality": False, "weekly_seasonality": False,
                   "daily_seasonality": False},
    },
    "Prophet-tuned": {
        "fn": "prophet",
        "params": {"seasonality_mode": "multiplicative", "n_changepoints": 5,
                   "changepoint_prior_scale": 0.1,
                   "yearly_seasonality": False, "weekly_seasonality": False,
                   "daily_seasonality": False},
    },
    "Theta-default": {"fn": "theta"},
    "SARIMAX-default": {"fn": "sarimax"},
}
if HAS_PMDARIMA:
    SERIES_CONFIGS["AutoARIMA"] = {"fn": "auto_arima"}



def _build_model_configs(cat_idx: list[int]) -> dict:
    """Фабрика: каждый вызов создаёт свежие (не обученные) экземпляры моделей."""
    return {
        "LinearRegression": LinearRegression(),
        "DT-shallow": DecisionTreeRegressor(max_depth=6, min_samples_leaf=5, random_state=42),
        "DT-deep": DecisionTreeRegressor(max_depth=12, min_samples_leaf=2, random_state=42),
        "DT-mid": DecisionTreeRegressor(max_depth=8, min_samples_leaf=4, random_state=42),
        "DT-conservative": DecisionTreeRegressor(
            max_depth=5, min_samples_leaf=10, min_samples_split=15, random_state=42,
        ),
        "DT-ccp": DecisionTreeRegressor(
            max_depth=15, min_samples_leaf=2, ccp_alpha=0.02, random_state=42,
        ),
        "CB-default": CatBoostRegressor(
            iterations=500, depth=6, learning_rate=0.1,
            verbose=0, random_state=42, cat_features=cat_idx,
        ),
        "CB-deep": CatBoostRegressor(
            iterations=800, depth=8, learning_rate=0.05,
            verbose=0, random_state=42, cat_features=cat_idx,
        ),
        "CB-conservative": CatBoostRegressor(
            iterations=300, depth=4, learning_rate=0.1,
            l2_leaf_reg=5, verbose=0, random_state=42, cat_features=cat_idx,
        ),
        "LGBM-default": LGBMRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            verbose=-1, random_state=42,
        ),
        "LGBM-tuned": LGBMRegressor(
            n_estimators=800, max_depth=8, learning_rate=0.05,
            num_leaves=63, min_child_samples=10, verbose=-1, random_state=42,
        ),
        "XGB-default": XGBRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            verbosity=0, random_state=42,
        ),
        "XGB-regularized": XGBRegressor(
            n_estimators=800, max_depth=8, learning_rate=0.05,
            reg_alpha=0.1, reg_lambda=5, verbosity=0, random_state=42,
        ),
        "RF-default": RandomForestRegressor(
            n_estimators=300, max_depth=12, min_samples_leaf=2,
            random_state=42, n_jobs=-1,
        ),
        "ET-default": ExtraTreesRegressor(
            n_estimators=300, max_depth=12, min_samples_leaf=2,
            random_state=42, n_jobs=-1,
        ),
        "HistGBM-default": HistGradientBoostingRegressor(
            max_iter=500, max_depth=6, learning_rate=0.1,
            random_state=42,
        ),
        "GBR-default": GradientBoostingRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            random_state=42,
        ),
    }

import sys
from pathlib import Path

try:
    _ROOT = Path(__file__).resolve().parent.parent
except NameError:
    _ROOT = Path("..").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
from scripts.calc_lag_features import (
    add_calc_shift_lags,
    calc_lag_columns_for_sets,
    DEFAULT_CALC_SHIFTS,
)

all_results: list = []
all_predictions: dict = {}
df_by_rid: dict = {}
mask_test_by_rid: dict = {}

for rid in RESIST_IDS:
    print(f"\n{'#' * 80}\n# resist_id={rid}\n{'#' * 80}")

    df = df_model39[df_model39["resist_id"] == rid].copy()
    df["op_code"] = df["operation_id"].astype("category").cat.codes

    group = df.groupby("operation_id")
    df["lag_1"] = group["fact_value"].shift(1)
    df["lag_12"] = group["fact_value"].shift(12)
    df["rolling_mean_3"] = group["fact_value"].transform(
        lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
    )
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].fillna(0)
    df = df.infer_objects(copy=False)

    miss = df.isnull().sum()
    miss = miss[miss > 0]
    if len(miss) > 0:
        print("Пропуски:", miss.to_dict())
        df = df.fillna(0)

    df, _lag_calc_cols, calc_keys_used = add_calc_shift_lags(df, drop_scalar_calc=True)
    b_flat, c_flat = calc_lag_columns_for_sets(calc_keys_used, whitelist=CALC_WHITELIST)
    print(
        f"calc_* lags {DEFAULT_CALC_SHIFTS}: ключей {len(calc_keys_used)}, "
        f"колонок набора B: {len(b_flat)}, набора C: {len(c_flat)}"
    )

    BASE_FEATURES = ["op_code", "month", "year", "workDayMonth_2", "calculate_minute"]
    FEATURE_SETS = {
        "A": BASE_FEATURES,
        "B": BASE_FEATURES + b_flat,
        "C": BASE_FEATURES + c_flat,
        "A+lags": BASE_FEATURES + LAG_FEATURES,
        "B+lags": BASE_FEATURES + b_flat + LAG_FEATURES,
        "C+lags": BASE_FEATURES + c_flat + LAG_FEATURES,
    }

    for fs_name, fs_cols in FEATURE_SETS.items():
        print(f"  {fs_name:>6}: {len(fs_cols)} признаков")

    total_tabular_expected = len(FEATURE_SETS) * 17

    mask_test = (df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS))
    mask_train = ~mask_test
    y_train = df.loc[mask_train, "fact_value"]
    y_test = df.loc[mask_test, "fact_value"]

    print(
        f"Train: {mask_train.sum()} строк, Test: {mask_test.sum()} строк | "
        f"операций: {df['operation_id'].nunique()}, строк: {len(df)}"
    )
    df_by_rid[rid] = df
    mask_test_by_rid[rid] = mask_test

    results_rid: list = []
    preds_rid: dict = {}

    n_test = int(mask_test.sum())
    t_fp, s_fp, q_fp = calc_metrics(y_test.values, df.loc[mask_test, "fact_predict"].values)
    print(f"fact_predict baseline: Точность={t_fp}% | Стабильность={s_fp}% | Качество={q_fp}%")

    if mask_train.sum() == 0:
        print(
            f"⚠ resist_id={rid}: train пуст (0 строк) — все модели получат predict=0. "
            f"Тестовое окно: year={TEST_YEAR}, months={TEST_MONTHS}."
        )
        y_zero = np.zeros(n_test)
        t_z, s_z, q_z = calc_metrics(y_test.values, y_zero)

        cat_idx_dummy = [0]
        model_configs_dummy = _build_model_configs(cat_idx_dummy)
        for fs_name, fs_cols in FEATURE_SETS.items():
            for config_name in model_configs_dummy:
                combo_key = f"{config_name} | {fs_name}"
                results_rid.append({
                    "resist_id": rid,
                    "Конфигурация": config_name,
                    "Набор признаков": fs_name,
                    "Тип": "табличная",
                    "Точность, %": t_z,
                    "Стабильность, %": s_z,
                    "Качество, %": q_z,
                    "Время, сек": 0.0,
                    "Точн. fact_predict, %": t_fp,
                    "Стаб. fact_predict, %": s_fp,
                    "Кач. fact_predict, %": q_fp,
                })
                preds_rid[combo_key] = y_zero

        for config_name in SERIES_CONFIGS:
            results_rid.append({
                "resist_id": rid,
                "Конфигурация": config_name,
                "Набор признаков": "—",
                "Тип": "per-series",
                "Точность, %": t_z,
                "Стабильность, %": s_z,
                "Качество, %": q_z,
                "Время, сек": 0.0,
                "Точн. fact_predict, %": t_fp,
                "Стаб. fact_predict, %": s_fp,
                "Кач. fact_predict, %": q_fp,
            })
            preds_rid[config_name] = y_zero

        print(f"  → записано {len(results_rid)} строк результатов (predict=0)")
        all_results.extend(results_rid)
        all_predictions[rid] = preds_rid
        continue

    combo_num = 0
    for fs_name, fs_cols in FEATURE_SETS.items():
        cat_idx = [fs_cols.index("op_code")]
        model_configs = _build_model_configs(cat_idx)
        X_train_fs = df.loc[mask_train, fs_cols]
        X_test_fs = df.loc[mask_test, fs_cols]

        print(f"\n{'=' * 90}")
        print(f"Набор: {fs_name} ({len(fs_cols)} признаков)")
        print(f"{'=' * 90}")
        print(f"{'Config':<22} | {'Accuracy':>8} | {'Stability':>9} | {'Quality':>7} | {'Time':>7}")
        print(f"{'-' * 90}")

        for config_name, model in model_configs.items():
            combo_num += 1
            combo_key = f"{config_name} | {fs_name}"
            t0 = time.perf_counter()
            model.fit(X_train_fs, y_train)
            y_pred = np.maximum(model.predict(X_test_fs), 0)
            elapsed = round(time.perf_counter() - t0, 2)
            tochnost, stab, kachestvo = calc_metrics(y_test.values, y_pred)
            results_rid.append({
                "resist_id": rid,
                "Конфигурация": config_name,
                "Набор признаков": fs_name,
                "Тип": "табличная",
                "Точность, %": tochnost,
                "Стабильность, %": stab,
                "Качество, %": kachestvo,
                "Время, сек": elapsed,
                "Точн. fact_predict, %": t_fp,
                "Стаб. fact_predict, %": s_fp,
                "Кач. fact_predict, %": q_fp,
            })
            preds_rid[combo_key] = y_pred
            print(f"{config_name:<22} | {tochnost:7.2f}% | {stab:8.2f}% | {kachestvo:6.2f}% | {elapsed:6.2f}s")

    print(f"\nТабличные комбинации для rid={rid}: {combo_num}/{total_tabular_expected}")

    print(f"\n{'=' * 80}")
    print(f"Per-series модели ({len(SERIES_CONFIGS)} конфигураций), rid={rid}")
    print(f"{'=' * 80}")

    n_tab = combo_num
    for config_name, cfg in SERIES_CONFIGS.items():
        t0 = time.perf_counter()
        if cfg["fn"] == "ets":
            y_pred = train_ets_per_series(
                df, mask_train, mask_test,
                trend=cfg["trend"],
                damped_trend=cfg["damped_trend"],
                seasonal=cfg.get("seasonal"),
                seasonal_periods=cfg.get("seasonal_periods"),
            )
        elif cfg["fn"] == "prophet":
            y_pred = train_prophet_per_series(df, mask_train, mask_test, cfg["params"])
            if y_pred is None:
                print(f"{config_name:<22} | пропущен (prophet не установлен)")
                continue
        elif cfg["fn"] == "theta":
            y_pred = train_theta_per_series(df, mask_train, mask_test)
        elif cfg["fn"] == "sarimax":
            y_pred = train_sarimax_per_series(df, mask_train, mask_test)
        elif cfg["fn"] == "auto_arima":
            y_pred = train_auto_arima_per_series(df, mask_train, mask_test)
        else:
            continue

        elapsed = round(time.perf_counter() - t0, 2)
        tochnost, stab, kachestvo = calc_metrics(y_test.values, y_pred)
        results_rid.append({
            "resist_id": rid,
            "Конфигурация": config_name,
            "Набор признаков": "—",
            "Тип": "per-series",
            "Точность, %": tochnost,
            "Стабильность, %": stab,
            "Качество, %": kachestvo,
            "Время, сек": elapsed,
            "Точн. fact_predict, %": t_fp,
            "Стаб. fact_predict, %": s_fp,
            "Кач. fact_predict, %": q_fp,
        })
        preds_rid[config_name] = y_pred
        print(f"{config_name:<22} | {tochnost:7.2f}% | {stab:8.2f}% | {kachestvo:6.2f}% | {elapsed:6.2f}s")

    print(
        f"\nВсего комбинаций для rid={rid}: {len(results_rid)} "
        f"(табличных: {n_tab}, per-series: {len(results_rid) - n_tab})"
    )

    all_results.extend(results_rid)
    all_predictions[rid] = preds_rid

results = all_results


#### Per-series модели: ETS, Prophet, Theta, SARIMAX (+ AutoARIMA при наличии pmdarima)

In [ ]:
# Определения train_* и SERIES_CONFIGS, а также цикл обучения — в предыдущей ячейке.
pass


#### Сводная таблица результатов

In [ ]:
df_results = pd.DataFrame(results)
df_results["Комбинация"] = df_results.apply(
    lambda r: f"{r['Конфигурация']} | {r['Набор признаков']}" if r["Набор признаков"] != "—" else r["Конфигурация"],
    axis=1,
)
df_results_full = df_results.sort_values(
    ["resist_id", "Качество, %"], ascending=[True, False]
).reset_index(drop=True)
df_results = (
    df_results_full.groupby("resist_id", group_keys=False)
    .head(SUMMARY_TOP_PER_RID)
    .reset_index(drop=True)
)
print(f"Строк в полной сводке: {len(df_results_full)}; в таблице ниже — топ-{SUMMARY_TOP_PER_RID} на каждый resist_id")
df_results


In [ ]:
TOP_N = SUMMARY_TOP_PER_RID
for rid in RESIST_IDS:
    sub = df_results_full[df_results_full["resist_id"] == rid].head(TOP_N)
    if sub.empty:
        continue
    fig, axes = plt.subplots(1, 3, figsize=(18, max(6, TOP_N * 0.45)))
    metrics_to_plot = ["Точность, %", "Стабильность, %", "Качество, %"]
    for ax, metric in zip(axes, metrics_to_plot):
        df_plot = sub.sort_values(metric, ascending=True)
        colors = ["#2196F3" if t == "табличная" else "#FF9800" for t in df_plot["Тип"]]
        ax.barh(df_plot["Комбинация"], df_plot[metric], color=colors)
        ax.set_xlabel(metric)
        ax.set_title(metric)
    plt.suptitle(f"TOP-{TOP_N} комбинаций (resist_id={rid})", fontsize=14)
    plt.tight_layout()
    plt.show()

for rid in RESIST_IDS:
    df_r = df_results_full[df_results_full["resist_id"] == rid]
    print(f"\n--- resist_id={rid}: лучший набор по каждой модели ---")
    for cfg_name in df_r["Конфигурация"].unique():
        best_row = df_r[df_r["Конфигурация"] == cfg_name].iloc[0]
        print(f"  {cfg_name:<22} → {best_row['Набор признаков']:<7} | Качество={best_row['Качество, %']:.2f}%")


#### Лучшая конфигурация: детали по операциям

In [ ]:
per_rid_best = {}

for rid in RESIST_IDS:
    df_r = df_results_full[df_results_full["resist_id"] == rid].sort_values(
        "Качество, %", ascending=False
    )
    if df_r.empty:
        continue
    best_row = df_r.iloc[0]
    best_combo = best_row["Комбинация"]
    best_config = best_row["Конфигурация"]
    best_fs = best_row["Набор признаков"]
    best_type = best_row["Тип"]
    print(f"\n{'='*60}\nresist_id={rid}\nЛучшая комбинация по Качеству: {best_combo} ({best_type})")
    print(f"  Модель: {best_config}, Набор: {best_fs}")
    print(
        f"  Точность={best_row['Точность, %']}% | Стабильность={best_row['Стабильность, %']}% | "
        f"Качество={best_row['Качество, %']}%"
    )

    best_pred = all_predictions[rid][best_combo]
    df = df_by_rid[rid]
    mask_test = mask_test_by_rid[rid]
    df_test = df.loc[mask_test].copy()
    df_test["predict_new"] = best_pred

    rows_op = []
    for (r_id, op_id), grp in df_test.groupby(["resist_id", "operation_id"]):
        t1, s1, q1 = calc_metrics(grp["fact_value"].values, grp["fact_predict"].values)
        t2, s2, q2 = calc_metrics(grp["fact_value"].values, grp["predict_new"].values)
        rows_op.append({
            "resist_id": r_id,
            "operation_id": op_id,
            "Точность (fact_predict)": t1,
            "Стабильность (fact_predict)": s1,
            "Качество (fact_predict)": q1,
            f"Точность ({best_combo})": t2,
            f"Стабильность ({best_combo})": s2,
            f"Качество ({best_combo})": q2,
        })

    df_metrics_op = pd.DataFrame(rows_op)
    df_detail = df_test[["resist_id", "operation_id", "year", "month",
                         "fact_value", "fact_predict", "predict_new"]].copy()
    df_detail = df_detail.sort_values(["resist_id", "operation_id", "year", "month"]).reset_index(drop=True)
    df_detail = df_detail.rename(columns={
        "fact_value": "fact_value (факт)",
        "fact_predict": "fact_predict (текущий)",
        "predict_new": f"{best_combo} (новый)",
    })
    per_rid_best[rid] = {
        "best_combo": best_combo,
        "df_metrics_op": df_metrics_op,
        "df_detail": df_detail,
    }
    print(f"Операций в тесте: {len(df_metrics_op)}")

if RESIST_IDS:
    r0 = RESIST_IDS[0]
    best_combo = per_rid_best[r0]["best_combo"]
    df_metrics_op = per_rid_best[r0]["df_metrics_op"]
    print("\n(первый id в списке: пример df_metrics_op.head())")
    print(df_metrics_op.head(10))


#### Экспорт в Excel

In [ ]:
from openpyxl import load_workbook

out_path = OUT_XLSX
out_path.parent.mkdir(parents=True, exist_ok=True)

sheet_triples = []
for rid in RESIST_IDS:
    df_itog = df_results_full[df_results_full["resist_id"] == rid].head(EXCEL_ITOG_TOP)
    if rid not in per_rid_best:
        continue
    b = per_rid_best[rid]
    sheet_triples.append((rid, df_itog, b["df_detail"], b["df_metrics_op"]))

names_to_remove = []
for rid, _, _, _ in sheet_triples:
    names_to_remove.extend([f"{rid} Итог", f"{rid} Данные теста", f"{rid} По операциям"])

if out_path.exists():
    wb = load_workbook(out_path)
    for sn in names_to_remove:
        if sn in wb.sheetnames:
            wb.remove(wb[sn])
    if len(wb.sheetnames) == 0:
        wb.create_sheet("_tmp")
    wb.save(out_path)
    wb.close()
    writer_mode = "a"
else:
    writer_mode = "w"

with pd.ExcelWriter(out_path, engine="openpyxl", mode=writer_mode) as w:
    for rid, df_itog, df_detail, df_metrics_op in sheet_triples:
        df_itog.to_excel(w, sheet_name=f"{rid} Итог", index=False)
        df_detail.to_excel(w, sheet_name=f"{rid} Данные теста", index=False)
        df_metrics_op.to_excel(w, sheet_name=f"{rid} По операциям", index=False)

wb = load_workbook(out_path)
if "_tmp" in wb.sheetnames and len(wb.sheetnames) > 1:
    wb.remove(wb["_tmp"])
    wb.save(out_path)
wb.close()

print(f"Excel: {out_path.resolve()}")
for rid, df_itog, _, _ in sheet_triples:
    print(f"  «{rid} Итог» ({len(df_itog)} строк), «{rid} Данные теста», «{rid} По операциям»")


#### Выводы

* Вариант ноутбука с **несколькими** `RESIST_IDS` за один прогон; полный перебор комбинаций на каждый id.
* В сводке в ноутбуке — **топ-5** по «Качеству» **на каждый** `resist_id`; полная таблица — `df_results_full`.
* Результаты в `model_comparison_multi_calc_lag12.xlsx`: на каждый id — три листа; лист **Итог** — **топ-20** строк (как в одно-id экспорте для `model_comparison_analytics.py`).
